# Notebook 5 — Dati per l'esperimento digits (MNIST / USPS / SVHN)

Scarica ed espone in formato numpy i tre dataset usati dai notebook 6-9:

- MNIST: 60.000 train + 10.000 test, 28x28 grayscale, 10 classi
- USPS: ~7.291 train + 2.007 test, 16x16 grayscale, 10 classi
- SVHN: ~73.257 train + ~26.032 test, 32x32 RGB, 10 classi (label "10"→"0",
  già gestita internamente da `torchvision.datasets.SVHN`, verificato sotto
  invece di assunto)

Il preprocessing comune (grayscale, resize a 32x32, normalizzazione sulle
statistiche del source) avviene più avanti, in `src/digits_data.py` -- qui
si salva solo la cache grezza convertita in `.npz`.

## Setup

In [ ]:
import sys
from pathlib import Path

cwd = Path().resolve()
PROJ = cwd
while not (PROJ / "src").exists():
    PROJ = PROJ.parent
sys.path.insert(0, str(PROJ))

import numpy as np
from torchvision import datasets

DATA_DIR = PROJ / "data" / "digits"
RAW_DIR = DATA_DIR / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)
print(f"PROJ={PROJ}")
print(f"DATA_DIR={DATA_DIR}")


def dump_split(dataset, out_path, remap_svhn_ten_to_zero: bool = False):
    X = np.stack([np.array(img) for img, _ in dataset]).astype(np.uint8)
    y = np.array([label for _, label in dataset], dtype=np.int64)
    if remap_svhn_ten_to_zero:
        # torchvision.datasets.SVHN.__init__ rimappa già internamente la
        # label 10 -> 0 (np.place(self.labels, self.labels == 10, 0)) --
        # verificato leggendo il suo sorgente, non assunto: qui si controlla
        # che il risultato sia davvero quello atteso, non lo si "corregge".
        assert not (y == 10).any(), "inattesa label 10 in SVHN dopo il remap interno di torchvision"
    np.savez_compressed(out_path, X=X, y=y)
    return X.shape, y.shape

def ensure_split(dataset_cls, split_kwargs, out_name: str, remap_svhn_ten_to_zero: bool = False):
    """Se out_name.npz esiste già, salta download e conversione (stampando
    un avviso) e ritorna le shape lette da lì; altrimenti costruisce il
    dataset (scarica se serve) e lo converte."""
    out_path = DATA_DIR / f"{out_name}.npz"
    if out_path.exists():
        print(f"{out_name}: {out_path} esiste già -- download e conversione saltati")
        d = np.load(out_path)
        return d["X"].shape, d["y"].shape
    ds = dataset_cls(**split_kwargs)
    return dump_split(ds, out_path, remap_svhn_ten_to_zero=remap_svhn_ten_to_zero)

## MNIST (28x28 grayscale, 10 classi)

In [ ]:
for split_name, train_flag in [("train", True), ("test", False)]:
    shape = ensure_split(datasets.MNIST, dict(root=str(RAW_DIR), train=train_flag, download=True),
                         f"mnist_{split_name}")
    print(f"mnist_{split_name}: X{shape[0]} y{shape[1]}")

## USPS (16x16 grayscale, 10 classi)

In [ ]:
for split_name, train_flag in [("train", True), ("test", False)]:
    shape = ensure_split(datasets.USPS, dict(root=str(RAW_DIR), train=train_flag, download=True),
                         f"usps_{split_name}")
    print(f"usps_{split_name}: X{shape[0]} y{shape[1]}")

## SVHN (32x32 RGB, 10 classi)

`ufldl.stanford.edu` (host di SVHN) può bloccarsi a metà download (HTTP
semplice, nessun timeout nel downloader di `torchvision`). Se la cella
sotto resta ferma per più di 1-2 minuti, interrompila e riprova il file
`.mat` bloccato con un retry con ripresa, es. per `train_32x32.mat`
(ripetere per `test_32x32.mat`, dimensioni attese 182.040.794 /
64.275.384 byte):

```bash
for i in $(seq 1 15); do
  curl -fSL -C - --connect-timeout 15 --max-time 200 --speed-time 20 --speed-limit 1000 \
    -o data/digits/raw/train_32x32.mat \
    "http://ufldl.stanford.edu/housenumbers/train_32x32.mat" && break
done
```

Poi rilancia la cella sotto: `torchvision` troverà il file già scaricato e
verificato (checksum) e passerà direttamente alla conversione in `.npz`.

In [ ]:
for split_name in ["train", "test"]:
    shape = ensure_split(datasets.SVHN, dict(root=str(RAW_DIR), split=split_name, download=True),
                         f"svhn_{split_name}", remap_svhn_ten_to_zero=True)
    print(f"svhn_{split_name}: X{shape[0]} y{shape[1]}")

## Conteggi finali

In [ ]:
for name in ["mnist_train", "mnist_test", "usps_train", "usps_test",
             "svhn_train", "svhn_test"]:
    d = np.load(DATA_DIR / f"{name}.npz")
    labels_present = sorted(set(d["y"].tolist()))
    print(f"  {name:12s}: {d['X'].shape[0]:6d} immagini, shape {d['X'].shape[1:]}, "
          f"label {labels_present}")